# 14. 낙엽송 진단 — 모델이 놓친 낙엽송은 어떤 픽셀인가 (재학습 없음)

**왜 하나:** 13a·13b 모두에서 침엽 라벨 낙엽송의 35–50%를 "침엽 아님"으로 놓쳤다. 다음 방향(라벨 손보기 / 덩어리 단위 / 새 정보)을 감이 아니라 근거로 고르려고 한다.

**무엇을 쓰나:** 13a에서 저장한 **v1+계절 모델(판정 통과 모델)의 픽셀 예측**(`exp6_folds`, 5시드) + 수종 래스터 + **임상도 shp(구획 번호·영급·밀도)**. 새로 학습하지 않는다.

**네 가지를 본다 (사전 기록 규칙으로 판정)**
1. **탄소 오차 나누기** — 인제·평창·금산의 +2%가 낙엽송 / 다른 침엽 / 활엽 / 혼효 중 어디서 나오나
2. **경계** — 놓친 낙엽송이 숲 구획 가장자리(섞인 픽셀)에 몰려 있나
3. **구획 통째** — 한 구획이 통째로 틀리나, 픽셀마다 흩어져 틀리나
4. **구획 나이·밀도** — 어린 숲·성긴 숲에서 더 놓치나

결과에 따라 다음 방향이 정해진다 (마지막 [10]이 자동으로 정리). 통째로 틀린 구획 목록(좌표 포함)도 저장한다 → 항공사진으로 직접 확인용.

**시간:** 약 15–30분 [추정] (지역마다 임상도 shp 읽기가 대부분 — dbf가 100MB 이상이라 필요한 열만 읽음). 지역마다 결과를 저장하므로 끊기면 **모두 실행** → 끝난 지역은 건너뜀.

**순서:** [1] 설정 → [2] 사전 기록 → [3] 지역별 준비(저장) → [4] 탄소 나누기 → [5] 경계 → [6] 구획 통째 → [7] 나이·밀도 → [8] 다른 침엽(금산) → [9] 생김새(기록) → [10] 요약

In [1]:
# [1] 설정
!pip install -q rasterio pyogrio

import os, glob, json, time, gc, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')
D        = '/content/drive/MyDrive/forest/data'
OUT      = '/content/drive/MyDrive/forest/outputs'
FOLD_DIR = f'{OUT}/exp6_folds'          # 13a 예측
DIAG_DIR = f'{OUT}/diag_larch'
os.makedirs(DIAG_DIR, exist_ok=True)

TRAIN_REGIONS = ['inje', 'hongcheon', 'gangneung', 'pyeongchang',
                 'bonghwa', 'geumsan', 'geochang', 'suncheon']
KR = {'inje': '인제', 'hongcheon': '홍천', 'gangneung': '강릉', 'pyeongchang': '평창',
      'bonghwa': '봉화', 'geumsan': '금산', 'geochang': '거창', 'suncheon': '순천'}
SEEDS = [16, 17, 18, 19, 20]
C  = {1: 79.25, 2: 101.02, 3: 92.01}
CV = np.array([0, C[1], C[2], C[3]])         # 인덱스 = 클래스 번호
LARCH_CODE = 13
GROUPS = ['낙엽송(침엽)', '다른 침엽', '활엽', '혼효']

EDGE_PX, INNER_PX = 2, 5        # 경계 = 가장자리에서 2칸(20m) 이내, 안쪽 = 5칸(50m) 이상
MIN_POLY_PX = 20                # 구획 분석에 쓰는 최소 표본 픽셀
WHOLE_MISS  = 0.7               # "통째로 틀린 구획" = 그 구획 낙엽송의 70% 이상 놓침

# 임상도 열 이름 (01번 출력에서 확인한 전국 임상도 형식). 02b와 같은 경로 규칙
WANT = ['KOFTR_GROU', 'KOFTR_NM', 'AGCLS_CD', 'AGCLS_NM', 'DNST_CD', 'DNST_NM', 'FROR_CD']
ATTR = {'age': 'AGCLS_CD', 'dns': 'DNST_CD', 'fror': 'FROR_CD'}     # 영급 / 밀도 / 임종(인공·천연)
ATTR_NM = {'age': 'AGCLS_NM', 'dns': 'DNST_NM', 'fror': None}

def find_shp(r):
    g = sorted(glob.glob(f'{D}/imsang/2025_{r}/*.shp'))
    assert len(g) == 1, f'{r}: imsang/2025_{r}/ 안에 shp {len(g)}개'
    return g[0]

import pyogrio
for r in TRAIN_REGIONS:
    f = find_shp(r)
    miss_cols = [c for c in WANT if c not in pyogrio.read_info(f)['fields']]
    if miss_cols: print(f'{KR[r]}: shp에 없는 열 {miss_cols} (해당 분석은 건너뜀)')
    n = len(SEEDS) - sum(os.path.exists(f'{FOLD_DIR}/s{s}_{r}.npz') for s in SEEDS)
    assert n == 0, f'{r}: 13a 폴드 파일 {n}개 없음'
print('shp·13a 폴드 확인 완료')

Mounted at /content/drive
shp·13a 폴드 확인 완료


In [2]:
# [2] 사전 기록 — 처음 실행할 때 diag_larch_prereg.json 저장 (이후 덮어쓰지 않음)
CONFIRMED = True
PREREG = {
    'date': time.strftime('%Y-%m-%d %H:%M'),
    'model': '13a v1+계절 (판정 통과 모델), 5시드 예측. 재학습 없음',
    'miss': '침엽 라벨 픽셀이 5시드 중 3번 이상 "침엽 아님"으로 예측되면 "놓침"',
    'H1_edge': {'measure': '낙엽송 가장자리에서의 거리(수종 래스터 기준)',
                'rule': '안쪽(≥5칸) 놓침률 ≤ 전체 놓침률의 절반 → "경계가 주원인"',
                'next': '덩어리 단위로 보기(56-3 B)'},
    'H2_polygon': {'measure': '표본 ≥20픽셀 구획만. 놓친 낙엽송 중 "구획 놓침률 ≥70%" 구획에 있는 비율. 대조 = 지역 안에서 놓침 표시를 섞은 값(20회 평균)',
                   'rule': '관측 ≥ 50% AND 관측 − 대조 ≥ 20%p → "구획 통째로 틀림"'},
    'H3_stand': {'measure': '영급별 놓침률 (8곳 합산, 낙엽송 픽셀의 5% 이상인 영급만)',
                 'rule': '최대 − 최소 ≥ 30%p → "구획 나이에 따라 크게 다름". 밀도는 같은 방식으로 기록만'},
    'mapping': {'H1': '경계 섞임 → 덩어리 단위(B)',
                'H2 & not H3': '구획 단위 원인, 나이로 설명 안 됨 → 라벨(구획) 의심 → 목록 구획을 항공사진으로 확인',
                'H2 & H3': '구획 상태(나이) 차이 → 진짜 숲 상태 차이 → 입력·클래스 정의',
                'none': '픽셀마다 흩어진 혼동 → 위성 정보로 구별 한계 → 새 정보(지형 등)'},
    'prediction': ['[추정] 인제·평창·금산 오차에서 낙엽송 몫이 절반 넘는 곳은 평창뿐. 금산은 "다른 침엽" 몫이 가장 큼',
                   '[추정] H1 아님 (안쪽 놓침률 > 전체의 절반)',
                   '[추정, 확신 낮음] H2 성립',
                   '[추정, 확신 낮음] H3 성립 (어린 낙엽송을 더 놓침)'],
    'not_used': '생김새(계절 NDVI) 비교는 기록만 — 놓친 픽셀은 모델이 보기에 활엽 같아서 놓친 것이라 판정에 쓰면 순환 논리',
}
pr = f'{OUT}/diag_larch_prereg.json'
if os.path.exists(pr):
    print('저장된 사전 기록 (덮어쓰지 않음):\n'); print(json.dumps(json.load(open(pr, encoding='utf-8')), ensure_ascii=False, indent=1))
elif CONFIRMED:
    json.dump(PREREG, open(pr, 'w', encoding='utf-8'), ensure_ascii=False, indent=1); print('사전 기록 저장:', pr)

사전 기록 저장: /content/drive/MyDrive/forest/outputs/diag_larch_prereg.json


In [3]:
# [3] 지역별 준비 → diag_larch/{지역}.npz, {지역}_poly.csv  (있으면 건너뜀)
assert os.path.exists(f'{OUT}/diag_larch_prereg.json'), '[2] 먼저'

for r in TRAIN_REGIONS:
    fz = f'{DIAG_DIR}/{r}.npz'
    if os.path.exists(fz):
        print(f'{KR[r]} 준비됨 — 건너뜀'); continue
    t0 = time.time()
    z2 = np.load(f'{OUT}/{r}_v2.npz')
    it = np.where(z2['ok'])[0]
    rr, cc, y = z2['rows'][it], z2['cols'][it], z2['y'][it]
    ndvi = z2['X2'][it][:, 20:25]                     # 계절별 NDVI 5개 (생김새 기록용)
    names2 = [str(s) for s in z2['names']]
    assert names2[20:25] == ['s1_ndvi', 's2_ndvi', 's3_ndvi', 's4_ndvi', 's5_ndvi'], names2[20:25]
    del z2

    # 수종 + 낙엽송 가장자리 거리
    with rasterio.open(f'{D}/{r}_species.tif') as s:
        sp = s.read(1); tf, crs, shp_ = s.transform, s.crs, s.shape
    sp_px = sp[rr, cc].astype(int)
    dist  = distance_transform_edt(sp == LARCH_CODE)[rr, cc].astype('float32')
    del sp; gc.collect()

    # 임상도 구획 번호 래스터화 — 02b build_labels와 같은 방식(전체 폴리곤, 같은 순서, 기본 설정)이라 수종 래스터와 그대로 맞아야 함
    path = find_shp(r)
    use = [c for c in WANT if c in pyogrio.read_info(path)['fields']]
    g = gpd.read_file(path, engine='pyogrio', columns=use)
    if g.crs is None or not g.crs.equals(crs.to_wkt()):
        g = g.to_crs(crs.to_wkt())
    pid = rasterize(((geom, i) for i, geom in enumerate(g.geometry)),
                    out_shape=shp_, transform=tf, fill=-1, dtype='int32')[rr, cc]
    has = pid >= 0
    sp_shp = pd.to_numeric(g['KOFTR_GROU'], errors='coerce').fillna(0).values
    agree = (sp_shp[pid[has]] == sp_px[has]).mean()
    print(f'{KR[r]}: 수종 래스터 vs shp 일치 {agree*100:.2f}%  (구획 번호 붙은 픽셀 {has.mean()*100:.1f}%)')
    assert agree > 0.99, f'{r}: 구획 래스터화가 기존 수종 래스터와 어긋남 — 멈추고 보고'
    if 'KOFTR_NM' in g:
        nm13 = g.loc[sp_shp == LARCH_CODE, 'KOFTR_NM'].astype(str).unique()[:3]
        assert any('낙엽송' in x for x in nm13), f'{r}: 코드 13 이름이 {nm13} — 낙엽송 코드 확인 필요'
    def lab(key):
        col = ATTR[key]
        if col not in g: return np.full(len(pid), 'NA')
        v = g[col].astype(str).replace({'None': 'NA', 'nan': 'NA'}).values
        return np.where(has, v[np.clip(pid, 0, None)], 'NA')
    age, dns, fror = lab('age'), lab('dns'), lab('fror')
    # 코드 → 이름 (표시용)
    nm = json.load(open(f'{DIAG_DIR}/names.json', encoding='utf-8')) if os.path.exists(f'{DIAG_DIR}/names.json') else {}
    for key, ncol, ccol in (('sp', 'KOFTR_NM', 'KOFTR_GROU'), ('age', 'AGCLS_NM', 'AGCLS_CD'), ('dns', 'DNST_NM', 'DNST_CD')):
        if ncol in g and ccol in g:
            d = nm.setdefault(key, {})
            for a, b in g[[ccol, ncol]].dropna().astype(str).drop_duplicates().values:
                d.setdefault(str(int(float(a))) if key == 'sp' and a.replace('.', '').isdigit() else a, b)
    json.dump(nm, open(f'{DIAG_DIR}/names.json', 'w', encoding='utf-8'), ensure_ascii=False)

    # 13a 예측 (5시드) → 놓침 횟수 + 탄소 오차 나누기
    larch = sp_px == LARCH_CODE
    grp = np.select([(y == 1) & larch, y == 1, y == 2, y == 3], [0, 1, 2, 3], -1)
    t_c = CV[y].sum()
    miss_cnt = np.zeros(len(y), 'int8'); contrib = np.zeros((len(SEEDS), 4))
    for i, s in enumerate(SEEDS):
        f = np.load(f'{FOLD_DIR}/s{s}_{r}.npz')
        p = f['sea_pred']
        assert len(p) == len(y), f'{r}: 13a 예측 길이 다름'
        miss_cnt += (p != 1)
        d = CV[p] - CV[y]
        contrib[i] = [d[grp == k].sum() / t_c * 100 for k in range(4)]
        assert abs(contrib[i].sum() - float(f['sea_eh'])) < 1e-3, f'{r} s{s}: 탄소 나누기 검산 실패'
    # 낙엽송 구획 목록 (항공사진 확인용)
    lp = has & (grp == 0)
    if lp.any():
        u, inv = np.unique(pid[lp], return_inverse=True)
        n_px  = np.bincount(inv)
        n_mis = np.bincount(inv, weights=(miss_cnt[lp] >= 3))
        cen = g.geometry.iloc[u].centroid.to_crs(4326)
        pd.DataFrame({'region': r, 'pid': u, 'n_px': n_px, 'miss_rate': n_mis / n_px,
                      'area_ha': g.geometry.iloc[u].area.values / 1e4,
                      'age': g['AGCLS_CD'].astype(str).values[u] if 'AGCLS_CD' in g else 'NA',
                      'density': g['DNST_CD'].astype(str).values[u] if 'DNST_CD' in g else 'NA',
                      'fror': g['FROR_CD'].astype(str).values[u] if 'FROR_CD' in g else 'NA',
                      'lat': cen.y.values, 'lon': cen.x.values}).to_csv(f'{DIAG_DIR}/{r}_poly.csv', index=False)
    np.savez_compressed(fz, y=y, grp=grp, sp=sp_px, pid=pid, dist=dist, age=age.astype(str), dns=dns.astype(str), fror=fror.astype(str),
                        miss_cnt=miss_cnt, contrib=contrib, ndvi=ndvi.astype('float32'))
    del g, pid, ndvi; gc.collect()
    print(f'{KR[r]} 저장 ({time.time()-t0:.0f}초)')

DZ = {r: dict(np.load(f'{DIAG_DIR}/{r}.npz')) for r in TRAIN_REGIONS}
print('\n8곳 준비 완료')

인제: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
인제 저장 (46초)
홍천: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
홍천 저장 (47초)
강릉: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
강릉 저장 (33초)
평창: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
평창 저장 (37초)
봉화: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
봉화 저장 (49초)
금산: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
금산 저장 (29초)
거창: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
거창 저장 (25초)
순천: 수종 래스터 vs shp 일치 100.00%  (구획 번호 붙은 픽셀 100.0%)
순천 저장 (24초)

8곳 준비 완료


In [4]:
# [4] ① 탄소 오차 나누기 — 13a v1+계절 딱 자르기 오차가 어느 픽셀에서 오나 (5시드 평균, %p)
rows = []
for r in TRAIN_REGIONS:
    c = DZ[r]['contrib'].mean(0); g = DZ[r]['grp']
    rows.append({'지역': KR[r], '전체 오차': c.sum(), **{k: c[i] for i, k in enumerate(GROUPS)},
                 '침엽중 낙엽송%': (g == 0).sum() / max((g <= 1).sum(), 1) * 100})
T4 = pd.DataFrame(rows); T4.to_csv(f'{OUT}/diag_larch_carbon.csv', index=False)
print(T4.round(2).to_string(index=False))
print('\n+ = 탄소를 많게 잡음 (침엽을 활엽·혼효로 → +). 네 칸의 합 = 전체 오차 (검산 통과)')
for r in ['inje', 'pyeongchang', 'geumsan']:
    c = DZ[r]['contrib'].mean(0)
    print(f'  {KR[r]}: 낙엽송 몫 {c[0]:+.2f} / 전체 {c.sum():+.2f} → {c[0]/c.sum()*100:.0f}%,  가장 큰 몫 = {GROUPS[int(np.argmax(c))]}')

지역  전체 오차  낙엽송(침엽)  다른 침엽    활엽    혼효  침엽중 낙엽송%
인제   2.16     0.86   2.04 -1.02  0.28     18.95
홍천   0.88     1.13   1.28 -1.57  0.05     37.37
강릉  -0.22     0.49   1.51 -1.84 -0.38     11.00
평창   2.22     1.75   1.42 -1.00  0.05     45.94
봉화   1.04     1.07   1.59 -1.42 -0.21     16.79
금산   2.17     0.72   2.74 -1.68  0.39     19.48
거창  -0.06     0.59   1.14 -1.26 -0.53     11.74
순천  -1.07     0.00   2.02 -2.71 -0.38      0.02

+ = 탄소를 많게 잡음 (침엽을 활엽·혼효로 → +). 네 칸의 합 = 전체 오차 (검산 통과)
  인제: 낙엽송 몫 +0.86 / 전체 +2.16 → 40%,  가장 큰 몫 = 다른 침엽
  평창: 낙엽송 몫 +1.75 / 전체 +2.22 → 79%,  가장 큰 몫 = 낙엽송(침엽)
  금산: 낙엽송 몫 +0.72 / 전체 +2.17 → 33%,  가장 큰 몫 = 다른 침엽


In [5]:
# [5] ② 경계 — 낙엽송 가장자리에서의 거리별 놓침률
rows, tot = [], np.zeros(6)
for r in TRAIN_REGIONS:
    z = DZ[r]; m = z['grp'] == 0
    if m.sum() == 0: continue
    miss, dist = z['miss_cnt'][m] >= 3, z['dist'][m]
    e, inn = dist <= EDGE_PX, dist >= INNER_PX
    rows.append({'지역': KR[r], '낙엽송 픽셀': int(m.sum()), '전체 놓침%': miss.mean() * 100,
                 '경계(≤20m) 비중%': e.mean() * 100, '경계 놓침%': miss[e].mean() * 100 if e.any() else np.nan,
                 '안쪽(≥50m) 놓침%': miss[inn].mean() * 100 if inn.any() else np.nan})
    tot += [miss.sum(), m.sum(), miss[e].sum(), e.sum(), miss[inn].sum(), inn.sum()]
T5 = pd.DataFrame(rows); T5.to_csv(f'{OUT}/diag_larch_edge.csv', index=False)
print(T5.round(1).to_string(index=False))
all_m, edge_m, inn_m = tot[0] / tot[1], tot[2] / tot[3], tot[4] / tot[5]
H1 = inn_m <= all_m / 2
print(f'\n8곳 합산: 전체 놓침 {all_m*100:.1f}% / 경계 {edge_m*100:.1f}% / 안쪽 {inn_m*100:.1f}%')
print(f'H1 (안쪽 ≤ 전체의 절반 {all_m*50:.1f}%): {"성립 — 경계가 주원인" if H1 else "불성립 — 안쪽에서도 놓친다"}')

지역  낙엽송 픽셀  전체 놓침%  경계(≤20m) 비중%  경계 놓침%  안쪽(≥50m) 놓침%
인제   51672    43.6          50.2    42.3          55.7
홍천   84355    35.7          55.1    38.2          34.6
강릉   26722    52.0          57.4    56.1          48.7
평창  121841    37.8          52.7    41.0          36.5
봉화   60288    48.1          72.7    48.5          56.1
금산   37976    51.3          62.6    53.4          50.3
거창   35526    42.5          44.2    47.5          37.6
순천      67    55.2          88.1    59.3           NaN

8곳 합산: 전체 놓침 42.1% / 경계 44.6% / 안쪽 42.2%
H1 (안쪽 ≤ 전체의 절반 21.1%): 불성립 — 안쪽에서도 놓친다


In [6]:
# [6] ③ 구획 통째로 틀리나 — 놓친 낙엽송 중 "70% 이상 놓친 구획"에 있는 비율 vs 무작위 대조
rng = np.random.default_rng(0)
def whole_share(pid, miss):
    u, inv = np.unique(pid, return_inverse=True)
    n = np.bincount(inv); k = np.bincount(inv, weights=miss)
    keep = n >= MIN_POLY_PX
    sel = keep[inv]
    rate = np.where(n > 0, k / np.maximum(n, 1), 0)
    in_whole = (rate >= WHOLE_MISS)[inv] & sel
    return (miss & in_whole).sum(), (miss & sel).sum()

rows, obs_t, null_t = [], np.zeros(2), np.zeros(2)
for r in TRAIN_REGIONS:
    z = DZ[r]; m = (z['grp'] == 0) & (z['pid'] >= 0)
    if m.sum() == 0: continue
    pid, miss = z['pid'][m], z['miss_cnt'][m] >= 3
    a, b = whole_share(pid, miss)
    nl = np.mean([whole_share(pid, rng.permutation(miss))[0] for _ in range(20)])
    obs_t += [a, b]; null_t += [nl, b]
    n_poly = int((np.bincount(np.unique(pid, return_inverse=True)[1]) >= MIN_POLY_PX).sum())
    rows.append({'지역': KR[r], '구획 수(≥20픽셀)': n_poly,
                 '관측%': a / max(b, 1) * 100, '무작위 대조%': nl / max(b, 1) * 100})
T6 = pd.DataFrame(rows); T6.to_csv(f'{OUT}/diag_larch_polygon.csv', index=False)
print(T6.round(1).to_string(index=False))
obs, nul = obs_t[0] / obs_t[1] * 100, null_t[0] / null_t[1] * 100
H2 = (obs >= 50) and (obs - nul >= 20)
print(f'\n8곳 합산: 관측 {obs:.1f}% vs 대조 {nul:.1f}%  (차이 {obs-nul:+.1f}%p)')
print(f'H2 (관측 ≥ 50% AND 차이 ≥ 20%p): {"성립 — 구획이 통째로 틀린다" if H2 else "불성립"}')

P = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{DIAG_DIR}/*_poly.csv'))])
top = (P[(P['n_px'] >= 30) & (P['miss_rate'] >= 0.9)]
       .sort_values('n_px', ascending=False).groupby('region').head(15))
top.to_csv(f'{OUT}/diag_larch_check_list.csv', index=False)
print(f'\n항공사진 확인 목록: {len(top)}개 구획 → outputs/diag_larch_check_list.csv (lat·lon 포함, 90% 이상 놓친 구획)')

지역  구획 수(≥20픽셀)  관측%  무작위 대조%
인제          519 61.6      0.1
홍천          927 41.4      0.0
강릉          301 60.9      1.1
평창         1366 42.7      0.0
봉화          698 55.0      0.6
금산          314 53.7      0.5
거창          153 45.5      0.0
순천            2 51.5      3.2

8곳 합산: 관측 49.7% vs 대조 0.2%  (차이 +49.5%p)
H2 (관측 ≥ 50% AND 차이 ≥ 20%p): 불성립

항공사진 확인 목록: 100개 구획 → outputs/diag_larch_check_list.csv (lat·lon 포함, 90% 이상 놓친 구획)


In [7]:
# [7] ④ 구획 나이(영급)·밀도별 놓침률 — 8곳 합산
def by_attr(key):
    a = np.concatenate([DZ[r][key][DZ[r]['grp'] == 0] for r in TRAIN_REGIONS])
    m = np.concatenate([DZ[r]['miss_cnt'][DZ[r]['grp'] == 0] >= 3 for r in TRAIN_REGIONS])
    t = pd.DataFrame({'값': a, 'miss': m}).groupby('값')['miss'].agg(['size', 'mean'])
    t['비중%'] = t['size'] / t['size'].sum() * 100; t['놓침%'] = t['mean'] * 100
    return t[['비중%', '놓침%']]

NM = json.load(open(f'{DIAG_DIR}/names.json', encoding='utf-8')) if os.path.exists(f'{DIAG_DIR}/names.json') else {}
H3 = False
for key, lab in (('age', '영급'), ('dns', '밀도 (A 소·B 중·C 밀)'), ('fror', '임종 (1 인공·2 천연, 기록)')):
    t = by_attr(key)
    if key in NM: t.index = [f'{i} {NM[key].get(i, "")}'.strip() for i in t.index]
    print(f'[{lab}]'); print(t.round(1).to_string()); print()
    t.to_csv(f'{OUT}/diag_larch_{key}.csv')
    v = t[(t['비중%'] >= 5) & (~t.index.str.startswith('NA'))]['놓침%']
    if key == 'age':
        if len(v) >= 2:
            H3 = (v.max() - v.min()) >= 30
            print(f'H3 (영급별 놓침률 최대−최소 {v.max()-v.min():.1f}%p ≥ 30): {"성립 — 구획 나이에 따라 크게 다름" if H3 else "불성립"}\n')
        else:
            print('H3 판단 불가 — 영급 열이 없거나 값이 하나뿐 (shp 열 목록을 채팅에 붙여 주세요)\n')

print('[지역 × 영급 놓침% (낙엽송 300픽셀 이상 칸만)]')
rows = []
for r in TRAIN_REGIONS:
    z = DZ[r]; m = z['grp'] == 0
    if m.sum() == 0: continue
    t = pd.DataFrame({'age': z['age'][m], 'miss': z['miss_cnt'][m] >= 3}).groupby('age')['miss'].agg(['size', 'mean'])
    rows.append(pd.Series({a: f'{v*100:.0f}' for a, (s, v) in t.iterrows() if s >= 300}, name=KR[r]))
print(pd.DataFrame(rows).fillna('').to_string())

[영급]
        비중%   놓침%
1 1영급  21.0  58.7
2 2영급   3.4  75.5
3 3영급   6.1  45.0
4 4영급  20.8  36.0
5 5영급  44.2  35.1
6 6영급   4.5  33.1
7 7영급   0.0  47.3

H3 (영급별 놓침률 최대−최소 23.6%p ≥ 30): 불성립

[밀도 (A 소·B 중·C 밀)]
      비중%   놓침%
A 소  22.6  60.3
B 중   3.8  53.5
C 밀  73.7  36.0

[임종 (1 인공·2 천연, 기록)]
     비중%   놓침%
값             
1  100.0  42.1

[지역 × 영급 놓침% (낙엽송 300픽셀 이상 칸만)]
     1   2   3   4   5   6
인제  78  75  32  27  27  30
홍천  64  79  37  30  26  30
강릉  60  75  55  48  48  30
평창  61  69  44  32  31  30
봉화  52  85  70  43  47  52
금산  54  86      59  45    
거창  44          48  43  35
순천                        


In [8]:
# [8] ⑤ 다른 침엽은 왜 놓치나 (금산 등) — 낙엽송 아닌 침엽의 수종별 놓침률 (기록)
SPN = (json.load(open(f'{DIAG_DIR}/names.json', encoding='utf-8')) if os.path.exists(f'{DIAG_DIR}/names.json') else {}).get('sp', {})
for r in TRAIN_REGIONS:
    z = DZ[r]; m = z['grp'] == 1
    t = pd.DataFrame({'수종코드': z['sp'][m], 'miss': z['miss_cnt'][m] >= 3}).groupby('수종코드')['miss'].agg(['size', 'mean'])
    t = t[t['size'] / t['size'].sum() >= 0.05].sort_values('size', ascending=False)
    print(f'{KR[r]} (다른 침엽 놓침 {(z["miss_cnt"][m] >= 3).mean()*100:.0f}%): ' +
          ', '.join(f'{SPN.get(str(c), "코드" + str(c))} {s/m.sum()*100:.0f}%→놓침{v*100:.0f}%' for c, (s, v) in t.iterrows()))
print('\n(다른 침엽 안에서 비중 5% 이상인 수종만. 이름 = 임상도 KOFTR_NM)')

인제 (다른 침엽 놓침 31%): 소나무 79%→놓침30%, 잣나무 16%→놓침19%
홍천 (다른 침엽 놓침 29%): 소나무 49%→놓침35%, 잣나무 48%→놓침21%
강릉 (다른 침엽 놓침 23%): 소나무 93%→놓침22%, 잣나무 6%→놓침35%
평창 (다른 침엽 놓침 31%): 소나무 57%→놓침30%, 잣나무 41%→놓침33%
봉화 (다른 침엽 놓침 17%): 소나무 92%→놓침16%, 잣나무 7%→놓침28%
금산 (다른 침엽 놓침 55%): 소나무 52%→놓침56%, 리기다소나무 29%→놓침42%, 잣나무 8%→놓침62%
거창 (다른 침엽 놓침 13%): 소나무 90%→놓침10%, 편백나무 6%→놓침62%
순천 (다른 침엽 놓침 25%): 소나무 52%→놓침19%, 편백나무 20%→놓침38%, 리기다소나무 17%→놓침25%, 곰솔 11%→놓침28%

(다른 침엽 안에서 비중 5% 이상인 수종만. 이름 = 임상도 KOFTR_NM)


In [9]:
# [9] ⑥ 생김새 (기록만, 판정에 안 씀) — 계절별 NDVI 중앙값, 8곳 합산
tags = ['3–4월', '5–6월', '7–8월', '9–10월', '11–12월']
sel = {'맞힌 낙엽송':   lambda z: (z['grp'] == 0) & (z['miss_cnt'] < 3),
       '놓친 낙엽송':   lambda z: (z['grp'] == 0) & (z['miss_cnt'] >= 3),
       '맞힌 다른침엽': lambda z: (z['grp'] == 1) & (z['miss_cnt'] < 3),
       '활엽':          lambda z: z['grp'] == 2}
T9 = pd.DataFrame({k: np.median(np.concatenate([DZ[r]['ndvi'][f(DZ[r])] for r in TRAIN_REGIONS]), 0) for k, f in sel.items()},
                  index=tags).T
T9.to_csv(f'{OUT}/diag_larch_look.csv')
print(T9.round(3).to_string())
print('\n읽는 법: "놓친 낙엽송" 줄이 "맞힌 낙엽송"과 "활엽" 중 어느 쪽에 가까운지 보는 참고용.')
print('놓친 픽셀은 원래 모델 눈에 활엽 같아서 놓친 것이라, 활엽에 가깝게 나오는 건 당연 → 판정에 쓰지 않음')

          3–4월   5–6월   7–8월  9–10월  11–12월
맞힌 낙엽송   0.366  0.852  0.860  0.845   0.398
놓친 낙엽송   0.367  0.875  0.889  0.860   0.426
맞힌 다른침엽  0.676  0.804  0.846  0.851   0.772
활엽       0.373  0.888  0.897  0.860   0.451

읽는 법: "놓친 낙엽송" 줄이 "맞힌 낙엽송"과 "활엽" 중 어느 쪽에 가까운지 보는 참고용.
놓친 픽셀은 원래 모델 눈에 활엽 같아서 놓친 것이라, 활엽에 가깝게 나오는 건 당연 → 판정에 쓰지 않음


In [10]:
# [10] 요약 — 사전 기록 규칙대로
print(f'H1 경계가 주원인        : {"성립" if H1 else "불성립"}')
print(f'H2 구획이 통째로 틀림    : {"성립" if H2 else "불성립"}')
print(f'H3 구획 나이에 따라 다름 : {"성립" if H3 else "불성립"}\n')
nxt = []
if H1: nxt.append('경계 섞임 → 덩어리 단위로 보기(B)')
if H2 and not H3: nxt.append('구획 단위 원인, 나이로 설명 안 됨 → 라벨(구획) 의심 → diag_larch_check_list.csv 구획을 항공사진으로 확인')
if H2 and H3: nxt.append('구획 상태(나이) 차이 → 진짜 숲 상태 차이 → 입력·클래스 정의')
if H3 and not H2: nxt.append('(기록) 나이에 따라 놓침률이 다르지만 구획 통째는 아님')
if not H1 and not H2: nxt.append('픽셀마다 흩어진 혼동 → 위성 정보로 구별 한계 → 새 정보(지형 등)')
print('사전 기록 대응 → ' + ' / '.join(nxt))
json.dump(dict(H1=bool(H1), H2=bool(H2), H3=bool(H3), next=nxt, date=time.strftime('%Y-%m-%d %H:%M')),
          open(f'{OUT}/diag_larch_judgement.json', 'w'), ensure_ascii=False, indent=1)
print('\n→ [4]–[10] 출력을 채팅에 붙여넣기')

H1 경계가 주원인        : 불성립
H2 구획이 통째로 틀림    : 불성립
H3 구획 나이에 따라 다름 : 불성립

사전 기록 대응 → 픽셀마다 흩어진 혼동 → 위성 정보로 구별 한계 → 새 정보(지형 등)

→ [4]–[10] 출력을 채팅에 붙여넣기
